In [21]:
import pandas as pd

df = pd.read_json("../data/events.jsonl", lines=True)
print(df.shape)
print(df.columns)

(1116, 13)
Index(['schema_version', 'event_id', 'session_id', 'timestamp_ms',
       'timestamp_iso', 'layer', 'event_type', 'source', 'context',
       'correlation', 'payload', 'metadata', 'extensions'],
      dtype='object')


In [22]:
app_switches=df[df['event_type']=='app_switch']
print(len(app_switches))
print(app_switches.iloc[0]['payload'])

365
{'new_app': {'app_name': 'WindowsTerminal', 'process_id': 19244, 'process_name': 'WindowsTerminal.exe', 'window_title': 'Theme M1 - Mixed Multi-Domain (HR / Finance / Ops)'}, 'previous_app': {'app_name': 'procmine-desktop-agent', 'process_id': 14204, 'process_name': 'procmine-desktop-agent.exe', 'window_title': 'ProcMine Agent'}, 'trigger': 'window_click'}


In [23]:
app_switches=app_switches.copy()
app_switches['process_id'] = app_switches['payload'].apply(lambda p: p['new_app']['process_id'])
app_switches['window_title'] = app_switches['payload'].apply(lambda p: p['new_app']['window_title'])

print(app_switches[['timestamp_iso', 'process_id', 'window_title']].head(10))

                      timestamp_iso  process_id  \
1  2026-06-30 12:19:54.465000+00:00       19244   
19 2026-06-30 12:20:11.414000+00:00       12660   
22 2026-06-30 12:20:12.436000+00:00        3924   
24 2026-06-30 12:20:12.886000+00:00        8628   
25 2026-06-30 12:20:12.915000+00:00       11780   
29 2026-06-30 12:20:15.204000+00:00       10552   
31 2026-06-30 12:20:17.210000+00:00       10552   
34 2026-06-30 12:21:12.403000+00:00        3924   
36 2026-06-30 12:21:12.970000+00:00        3924   
37 2026-06-30 12:21:13.067000+00:00        3924   

                                         window_title  
1   Theme M1 - Mixed Multi-Domain (HR / Finance / ...  
19                                            Notepad  
22                           Untitled - Google Chrome  
24                           Untitled - Google Chrome  
25                           Untitled - Google Chrome  
29                                    Opening - Excel  
31                                            

In [24]:
#Noise filter#
def filter_noise(df):
    kept_rows=[]
    last_identity=None
    for idx,row in df.iterrows():
         identity=(row['process_id'],row['window_title'])
         if(identity!=last_identity):
             kept_rows.append(row)
             last_identity=identity

    return pd.DataFrame(kept_rows)
clean = filter_noise(app_switches)
print(f"Before: {len(app_switches)} rows, After: {len(clean)} rows")
print(clean[['timestamp_iso', 'process_id', 'window_title']].head(20))

Before: 365 rows, After: 55 rows
                       timestamp_iso  process_id  \
1   2026-06-30 12:19:54.465000+00:00       19244   
19  2026-06-30 12:20:11.414000+00:00       12660   
22  2026-06-30 12:20:12.436000+00:00        3924   
24  2026-06-30 12:20:12.886000+00:00        8628   
25  2026-06-30 12:20:12.915000+00:00       11780   
29  2026-06-30 12:20:15.204000+00:00       10552   
31  2026-06-30 12:20:17.210000+00:00       10552   
34  2026-06-30 12:21:12.403000+00:00        3924   
67  2026-06-30 12:21:20.116000+00:00       10552   
93  2026-06-30 12:21:31.523000+00:00       12660   
115 2026-06-30 12:21:40.942000+00:00        8628   
150 2026-06-30 12:21:50.882000+00:00       21888   
154 2026-06-30 12:21:53.014000+00:00       21888   
174 2026-06-30 12:22:13.439000+00:00        8628   
177 2026-06-30 12:22:14.599000+00:00       12660   
193 2026-06-30 12:22:17.992000+00:00        8628   
219 2026-06-30 12:22:31.095000+00:00       10552   
263 2026-06-30 12:23:35.349000+

In [31]:
#Factor-2
ANCHOR_SYSTEMS = ['財務会計システム', 'HR人事給与システム', '受発注在庫管理システム', 'ProcMine SSO']
from datetime import datetime, timezone
def get_anchor(title):
    for a in ANCHOR_SYSTEMS:
        if a in title:
            return a
    return None

def ts_to_ms(row):
    return int(row['timestamp_iso'].timestamp() * 1000)

segments = []
seg_start_ms = ts_to_ms(clean.iloc[0])
current_anchor = None

for i in range(len(clean)):
    title = clean.iloc[i]['window_title']
    anchor = get_anchor(title)
    if anchor is not None:
        if current_anchor is None:
            current_anchor = anchor
        elif anchor != current_anchor:
            seg_end_ms = ts_to_ms(clean.iloc[i])
            segments.append({'start_ms': seg_start_ms, 'end_ms': seg_end_ms})
            seg_start_ms = seg_end_ms
            current_anchor = anchor

session_end_ms = int(df['timestamp_iso'].max().timestamp() * 1000)
segments.append({'start_ms': seg_start_ms, 'end_ms': session_end_ms})

print(f"Generated {len(segments)} segments")
for s in segments:
    print(s)

for s in segments:
    start = datetime.fromtimestamp(s['start_ms']/1000, tz=timezone.utc)
    end = datetime.fromtimestamp(s['end_ms']/1000, tz=timezone.utc)
    dur = (end-start).total_seconds()
    print(f"{start.strftime('%H:%M:%S')} -> {end.strftime('%H:%M:%S')}  (duration: {dur:.0f}s)")

Generated 12 segments
{'start_ms': 1782821994465, 'end_ms': 1782822100942}
{'start_ms': 1782822100942, 'end_ms': 1782822225607}
{'start_ms': 1782822225607, 'end_ms': 1782822260531}
{'start_ms': 1782822260531, 'end_ms': 1782822362215}
{'start_ms': 1782822362215, 'end_ms': 1782822392097}
{'start_ms': 1782822392097, 'end_ms': 1782822415105}
{'start_ms': 1782822415105, 'end_ms': 1782822457804}
{'start_ms': 1782822457804, 'end_ms': 1782822507862}
{'start_ms': 1782822507862, 'end_ms': 1782822509611}
{'start_ms': 1782822509611, 'end_ms': 1782822513680}
{'start_ms': 1782822513680, 'end_ms': 1782822523663}
{'start_ms': 1782822523663, 'end_ms': 1782822597781}
12:19:54 -> 12:21:40  (duration: 106s)
12:21:40 -> 12:23:45  (duration: 125s)
12:23:45 -> 12:24:20  (duration: 35s)
12:24:20 -> 12:26:02  (duration: 102s)
12:26:02 -> 12:26:32  (duration: 30s)
12:26:32 -> 12:26:55  (duration: 23s)
12:26:55 -> 12:27:37  (duration: 43s)
12:27:37 -> 12:28:27  (duration: 50s)
12:28:27 -> 12:28:29  (duration: 2s

In [32]:
##Factor-3
import json
import pandas as pd

def load_events_file(path):
    events = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                events.append(json.loads(line))
    return events

# Load both chunks separately
chunk1_events = load_events_file('../data/events.jsonl')       # chunk_1200
chunk2_events = load_events_file('../data/events (1).jsonl')   # chunk_1230

print(f"Chunk 1: {len(chunk1_events)} events")
print(f"Chunk 2: {len(chunk2_events)} events")

# Merge into one list, then sort by REAL timestamp (not file order)
all_events = chunk1_events + chunk2_events
all_events.sort(key=lambda e: e['timestamp_ms'])

print(f"Combined: {len(all_events)} events")
print(f"First event: {all_events[0]['timestamp_iso']}")
print(f"Last event:  {all_events[-1]['timestamp_iso']}")

Chunk 1: 1116 events
Chunk 2: 1232 events
Combined: 2348 events
First event: 2026-06-30T12:19:53.821Z
Last event:  2026-06-30T12:45:08.101Z


In [37]:
df=pd.DataFrame(all_events)
df['timestamp_iso'] = pd.to_datetime(df['timestamp_iso'])
for s in segments:
    start = datetime.fromtimestamp(s['start_ms']/1000, tz=timezone.utc)
    end = datetime.fromtimestamp(s['end_ms']/1000, tz=timezone.utc)
    dur = (end-start).total_seconds()
    print(f"{start.strftime('%H:%M:%S')} -> {end.strftime('%H:%M:%S')}  (duration: {dur:.0f}s)")
app_switches=df[df['event_type']=='app_switch']
print(len(app_switches))
print(app_switches.iloc[0]['payload'])

app_switches=app_switches.copy()
app_switches['process_id'] = app_switches['payload'].apply(lambda p: p['new_app']['process_id'])
app_switches['window_title'] = app_switches['payload'].apply(lambda p: p['new_app']['window_title'])

print(app_switches[['timestamp_iso', 'process_id', 'window_title']].head(10))

#Noise filter#
def filter_noise(df):
    kept_rows=[]
    last_identity=None
    for idx,row in df.iterrows():
         identity=(row['process_id'],row['window_title'])
         if(identity!=last_identity):
             kept_rows.append(row)
             last_identity=identity

    return pd.DataFrame(kept_rows)
clean = filter_noise(app_switches)
print(f"Before: {len(app_switches)} rows, After: {len(clean)} rows")
print(clean[['timestamp_iso', 'process_id', 'window_title']].head(20))

#Factor-2
ANCHOR_SYSTEMS = ['財務会計システム', 'HR人事給与システム', '受発注在庫管理システム', 'ProcMine SSO']
from datetime import datetime, timezone
def get_anchor(title):
    for a in ANCHOR_SYSTEMS:
        if a in title:
            return a
    return None

def ts_to_ms(row):
    return int(row['timestamp_iso'].timestamp() * 1000)

segments = []
seg_start_ms = ts_to_ms(clean.iloc[0])
current_anchor = None

for i in range(len(clean)):
    title = clean.iloc[i]['window_title']
    anchor = get_anchor(title)
    if anchor is not None:
        if current_anchor is None:
            current_anchor = anchor
        elif anchor != current_anchor:
            seg_end_ms = ts_to_ms(clean.iloc[i])
            segments.append({'start_ms': seg_start_ms, 'end_ms': seg_end_ms})
            seg_start_ms = seg_end_ms
            current_anchor = anchor

session_end_ms = int(df['timestamp_iso'].max().timestamp() * 1000)
segments.append({'start_ms': seg_start_ms, 'end_ms': session_end_ms})

print(f"Generated {len(segments)} segments")
for s in segments:
    print(s)

for s in segments:
    start = datetime.fromtimestamp(s['start_ms']/1000, tz=timezone.utc)
    end = datetime.fromtimestamp(s['end_ms']/1000, tz=timezone.utc)
    dur = (end-start).total_seconds()
    print(f"{start.strftime('%H:%M:%S')} -> {end.strftime('%H:%M:%S')}  (duration: {dur:.0f}s)")

671
{'new_app': {'app_name': 'WindowsTerminal', 'process_id': 19244, 'process_name': 'WindowsTerminal.exe', 'window_title': 'Theme M1 - Mixed Multi-Domain (HR / Finance / Ops)'}, 'previous_app': {'app_name': 'procmine-desktop-agent', 'process_id': 14204, 'process_name': 'procmine-desktop-agent.exe', 'window_title': 'ProcMine Agent'}, 'trigger': 'window_click'}
                      timestamp_iso  process_id  \
2  2026-06-30 12:19:54.465000+00:00       19244   
19 2026-06-30 12:20:11.414000+00:00       12660   
23 2026-06-30 12:20:12.436000+00:00        3924   
24 2026-06-30 12:20:12.886000+00:00        8628   
25 2026-06-30 12:20:12.915000+00:00       11780   
29 2026-06-30 12:20:15.204000+00:00       10552   
31 2026-06-30 12:20:17.210000+00:00       10552   
34 2026-06-30 12:21:12.403000+00:00        3924   
36 2026-06-30 12:21:12.970000+00:00        3924   
37 2026-06-30 12:21:13.067000+00:00        3924   

                                         window_title  
2   Theme M1 - Mixe

In [39]:
# --- Evaluate the FULL SESSION against ground truth ---

with open('../data/gt_manifest.json', encoding='utf-8') as f:
    manifest = json.load(f)

true_segments = []
for proc in manifest['processes']:
    for ex in proc['executions']:
        if ex.get('start_ts') and ex.get('end_ts'):  # skip incomplete entries
            true_segments.append({
                'process_code': proc['code'],
                'case_id': ex['case_id'],
                'start_ts': ex['start_ts'],
                'end_ts': ex['end_ts'],
            })

def to_ms(ts):
    return int(datetime.fromisoformat(ts).timestamp() * 1000)

def iou_ms(pred_start, pred_end, true_start, true_end):
    inter = max(0, min(pred_end, true_end) - max(pred_start, true_start))
    union = max(pred_end, true_end) - min(pred_start, true_start)
    return inter / union if union > 0 else 0.0

scores = []
for t in true_segments:
    t_start, t_end = to_ms(t['start_ts']), to_ms(t['end_ts'])
    best_iou = max((iou_ms(s['start_ms'], s['end_ms'], t_start, t_end) for s in segments), default=0.0)
    scores.append({'process': t['process_code'], 'case': t['case_id'], 'iou': best_iou})

avg_iou = sum(s['iou'] for s in scores) / len(scores) if scores else 0
print(f"Evaluated {len(scores)} true segments across the full session")
print(f"Average IoU: {avg_iou:.3f}\n")
for s in sorted(scores, key=lambda s: s['iou']):
    print(f"  {s['process']:3s} {s['case']:15s} IoU={s['iou']:.3f}")

Evaluated 28 true segments across the full session
Average IoU: 0.378

  G   EXP-175009-002  IoU=0.083
  J   PM-175009-001   IoU=0.085
  E   OB-175009-001   IoU=0.101
  A   RT-175009-003   IoU=0.102
  A   RT-175009-004   IoU=0.150
  C   LA-175009-003   IoU=0.160
  B   PI-175009-002   IoU=0.172
  A   RT-175009-001   IoU=0.188
  J   PM-175009-002   IoU=0.191
  O   RET-175009-003  IoU=0.194
  G   EXP-175009-003  IoU=0.214
  E   OB-175009-002   IoU=0.221
  J   PM-175009-003   IoU=0.229
  B   PI-175009-003   IoU=0.246
  H   BR-175009-001   IoU=0.263
  C   LA-175009-004   IoU=0.313
  C   LA-175009-002   IoU=0.323
  C   LA-175009-001   IoU=0.356
  G   EXP-175009-004  IoU=0.363
  H   BR-175009-004   IoU=0.371
  B   PI-175009-001   IoU=0.452
  E   OB-175009-003   IoU=0.462
  M   SUP-175009-004  IoU=0.478
  A   RT-175009-002   IoU=0.967
  O   RET-175009-001  IoU=0.975
  H   BR-175009-002   IoU=0.977
  M   SUP-175009-001  IoU=0.978
  M   SUP-175009-002  IoU=0.982


In [48]:
CLIPBOARD_MIN_GAP_S = 5      # ignore clipboard events too close together (same copy/paste action)
LONG_SEGMENT_THRESHOLD_MS = 60000   # only apply clipboard-splitting inside segments longer than 60s

# Get all clipboard_change events with their timestamps (already have df loaded with all_events)
clipboard_events = df[df['event_type'] == 'clipboard_change'].copy()
clipboard_events = clipboard_events.sort_values('timestamp_iso')
clipboard_ts_list = [ts_to_ms({'timestamp_iso': t}) for t in clipboard_events['timestamp_iso']]

def refine_with_clipboard(segments, clipboard_ts_list):
    refined = []
    for seg in segments:
        duration = seg['end_ms'] - seg['start_ms']
        if duration <= LONG_SEGMENT_THRESHOLD_MS:
            refined.append(seg)
            continue
        
        # find clipboard events strictly inside this segment
        inside = [t for t in clipboard_ts_list if seg['start_ms'] < t < seg['end_ms']]
        
        # drop clipboard events too close to each other (keep first of each cluster)
        split_points = []
        last_kept = None
        for t in inside:
            if last_kept is None or (t - last_kept) > CLIPBOARD_MIN_GAP_S * 1000:
                split_points.append(t)
                last_kept = t
        
        if not split_points:
            refined.append(seg)
            continue
        
        # break the segment at each split point
        prev = seg['start_ms']
        for sp in split_points:
            refined.append({'start_ms': prev, 'end_ms': sp})
            prev = sp
        refined.append({'start_ms': prev, 'end_ms': seg['end_ms']})
    return refined

refined_segments = refine_with_clipboard(segments, clipboard_ts_list)
print(f"Before refinement: {len(segments)} segments")
print(f"After clipboard refinement: {len(refined_segments)} segments")

scores = []
for t in true_segments:   # <-- loop over TRUE segments, not refined_segments
    t_start, t_end = to_ms(t['start_ts']), to_ms(t['end_ts'])
    best_iou = max((iou_ms(s['start_ms'], s['end_ms'], t_start, t_end) for s in refined_segments), default=0.0)
    scores.append({'process': t['process_code'], 'case': t['case_id'], 'iou': best_iou})

avg_iou = sum(s['iou'] for s in scores) / len(scores) if scores else 0
print(f"Evaluated {len(scores)} true segments across the full session")
print(f"Average IoU: {avg_iou:.3f}\n")
for s in sorted(scores, key=lambda s: s['iou']):
    print(f"  {s['process']:3s} {s['case']:15s} IoU={s['iou']:.3f}")

Before refinement: 19 segments
After clipboard refinement: 44 segments
Evaluated 28 true segments across the full session
Average IoU: 0.560

  B   PI-175009-002   IoU=0.224
  O   RET-175009-003  IoU=0.274
  A   RT-175009-001   IoU=0.280
  H   BR-175009-001   IoU=0.289
  E   OB-175009-002   IoU=0.296
  J   PM-175009-002   IoU=0.317
  E   OB-175009-001   IoU=0.338
  J   PM-175009-003   IoU=0.350
  H   BR-175009-004   IoU=0.371
  G   EXP-175009-002  IoU=0.396
  A   RT-175009-003   IoU=0.413
  A   RT-175009-004   IoU=0.446
  C   LA-175009-004   IoU=0.515
  G   EXP-175009-003  IoU=0.542
  J   PM-175009-001   IoU=0.547
  G   EXP-175009-004  IoU=0.555
  B   PI-175009-001   IoU=0.565
  M   SUP-175009-004  IoU=0.576
  C   LA-175009-003   IoU=0.640
  E   OB-175009-003   IoU=0.693
  C   LA-175009-002   IoU=0.698
  B   PI-175009-003   IoU=0.703
  C   LA-175009-001   IoU=0.766
  A   RT-175009-002   IoU=0.967
  O   RET-175009-001  IoU=0.975
  H   BR-175009-002   IoU=0.977
  M   SUP-175009-001  IoU=

In [56]:
segments = []
seg_start_ms = ts_to_ms(clean.iloc[0])
current_anchor = None

for i in range(len(clean)):
    title = clean.iloc[i]['window_title']
    anchor = get_anchor(title)
    if anchor is not None:
        if current_anchor is None:
            current_anchor = anchor
        elif anchor != current_anchor:
            seg_end_ms = ts_to_ms(clean.iloc[i])
            segments.append({'start_ms': seg_start_ms, 'end_ms': seg_end_ms, 'label': current_anchor})  # <-- added label
            seg_start_ms = seg_end_ms
            current_anchor = anchor

session_end_ms = int(df['timestamp_iso'].max().timestamp() * 1000)
segments.append({'start_ms': seg_start_ms, 'end_ms': session_end_ms, 'label': current_anchor})  # <-- added label

print(f"Generated {len(segments)} segments")
for s in segments:
    print(s)
def refine_with_clipboard(segments, clipboard_ts_list):
    refined = []
    for seg in segments:
        duration = seg['end_ms'] - seg['start_ms']
        label = seg.get('label', 'unknown')  # <-- carry label forward
        if duration <= LONG_SEGMENT_THRESHOLD_MS:
            refined.append(seg)
            continue

        inside = [t for t in clipboard_ts_list if seg['start_ms'] < t < seg['end_ms']]
        split_points = []
        last_kept = None
        for t in inside:
            if last_kept is None or (t - last_kept) > CLIPBOARD_MIN_GAP_S * 1000:
                split_points.append(t)
                last_kept = t

        if not split_points:
            refined.append(seg)
            continue

        prev = seg['start_ms']
        for sp in split_points:
            refined.append({'start_ms': prev, 'end_ms': sp, 'label': label})  # <-- pass label along
            prev = sp
        refined.append({'start_ms': prev, 'end_ms': seg['end_ms'], 'label': label})  # <-- pass label along
    return refined

refined_segments = refine_with_clipboard(segments, clipboard_ts_list)
print(f"After refinement: {len(refined_segments)} segments, all labeled")
for s in refined_segments[:10]:
    print(s)

Generated 19 segments
{'start_ms': 1782821994465, 'end_ms': 1782822100942, 'label': '財務会計システム'}
{'start_ms': 1782822100942, 'end_ms': 1782822225607, 'label': 'HR人事給与システム'}
{'start_ms': 1782822225607, 'end_ms': 1782822260531, 'label': '受発注在庫管理システム'}
{'start_ms': 1782822260531, 'end_ms': 1782822362215, 'label': 'HR人事給与システム'}
{'start_ms': 1782822362215, 'end_ms': 1782822392097, 'label': '財務会計システム'}
{'start_ms': 1782822392097, 'end_ms': 1782822415105, 'label': 'HR人事給与システム'}
{'start_ms': 1782822415105, 'end_ms': 1782822457804, 'label': '受発注在庫管理システム'}
{'start_ms': 1782822457804, 'end_ms': 1782822507862, 'label': '財務会計システム'}
{'start_ms': 1782822507862, 'end_ms': 1782822509611, 'label': 'HR人事給与システム'}
{'start_ms': 1782822509611, 'end_ms': 1782822513680, 'label': 'ProcMine SSO'}
{'start_ms': 1782822513680, 'end_ms': 1782822523663, 'label': '財務会計システム'}
{'start_ms': 1782822523663, 'end_ms': 1782822794373, 'label': 'HR人事給与システム'}
{'start_ms': 1782822794373, 'end_ms': 1782822868352, 'label': '受発注在庫管理

In [57]:
# --- Factor 7: variant detection using action density ---

def count_actions_in_range(df, start_ms, end_ms):
    """Count keystrokes/clicks/shortcuts/scrolls within a time range."""
    subset = df[(df['timestamp_iso'].apply(lambda t: int(t.timestamp()*1000)) >= start_ms) &
                (df['timestamp_iso'].apply(lambda t: int(t.timestamp()*1000)) <= end_ms)]
    counts = subset['event_type'].value_counts()
    total = (counts.get('keystroke', 0) + counts.get('mouse_click', 0) +
             counts.get('shortcut', 0) + counts.get('mouse_scroll', 0))
    return total

# Compute action density for every segment
for seg in refined_segments:
    duration_s = (seg['end_ms'] - seg['start_ms']) / 1000
    action_count = count_actions_in_range(df, seg['start_ms'], seg['end_ms'])
    seg['action_density'] = action_count / duration_s if duration_s > 0 else 0

# Group by label, compute median density per label
import statistics
from collections import defaultdict

density_by_label = defaultdict(list)
for seg in refined_segments:
    density_by_label[seg['label']].append(seg['action_density'])

median_density = {label: statistics.median(densities) for label, densities in density_by_label.items()}

# Flag segments with notably higher-than-median density as "exception" variant
EXCEPTION_MULTIPLIER = 1.8   # threshold: 80% above the median for that label

for seg in refined_segments:
    med = median_density[seg['label']]
    if med > 0 and seg['action_density'] > med * EXCEPTION_MULTIPLIER:
        seg['label'] = seg['label'] + '_exception'

print("Segments after variant tagging:")
for s in refined_segments:
    print(f"  {s['label']:30s} density={s['action_density']:.2f}")

Segments after variant tagging:
  財務会計システム                       density=0.38
  財務会計システム                       density=1.09
  HR人事給与システム                     density=0.52
  HR人事給与システム                     density=1.12
  HR人事給与システム                     density=0.51
  HR人事給与システム                     density=0.20
  受発注在庫管理システム                    density=0.77
  HR人事給与システム                     density=0.64
  HR人事給与システム                     density=0.40
  財務会計システム                       density=1.17
  HR人事給与システム                     density=1.22
  受発注在庫管理システム                    density=0.63
  財務会計システム                       density=1.30
  HR人事給与システム                     density=0.00
  ProcMine SSO                   density=9.58
  財務会計システム_exception             density=2.70
  HR人事給与システム                     density=1.07
  HR人事給与システム                     density=1.29
  HR人事給与システム                     density=0.47
  HR人事給与システム_exception           density=1.38
  HR人事給与システム                     density=0.74
  

In [58]:
import shutil
from pathlib import Path

source = Path(r"G:\.shortcut-targets-by-id\1WSBZQxOODKFDzYBkIik5g90NWKS7Gwl0\AI Engineer\dataset_a")
destination = Path(r"C:\Users\xlaks\OneDrive\Desktop\intern-assignment-2026\data\dataset_a")

destination.mkdir(parents=True, exist_ok=True)

copied = 0
for file_path in source.rglob("*"):
    if file_path.is_file() and "screenshots" not in file_path.parts:
        rel_path = file_path.relative_to(source)
        dest_path = destination / rel_path
        dest_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file_path, dest_path)
        copied += 1
        if copied % 50 == 0:
            print(f"Copied {copied} files so far...")

print(f"Done. Copied {copied} files total.")

Copied 50 files so far...
Copied 100 files so far...
Copied 150 files so far...
Copied 200 files so far...
Copied 250 files so far...
Copied 300 files so far...
Copied 350 files so far...
Copied 400 files so far...
Copied 450 files so far...
Copied 500 files so far...
Copied 550 files so far...
Done. Copied 557 files total.


In [59]:
from pathlib import Path

dataset_a_path = Path("../data/dataset_a")
session_dirs = sorted([d for d in dataset_a_path.iterdir() if d.is_dir()])
print(f"Found {len(session_dirs)} session folders")

# Spot check the first one
first_session = session_dirs[0]
print(f"\nContents of {first_session.name}:")
for item in first_session.iterdir():
    print(" ", item.name)

Found 63 session folders

Contents of ses_20260630-121953-LAPTOP-R36BQBTE:
  chunk_1200
  chunk_1230
  chunk_20260630-1200-LAPTOP-R36BQBTE
  chunk_20260630-1230-LAPTOP-R36BQBTE
  desktop.ini
  gt.jsonl
  gt_manifest.json


In [61]:
import json
import pandas as pd
import statistics
from collections import defaultdict
from pathlib import Path
from datetime import datetime

# ============================================================
# FULL PIPELINE - all factors combined into reusable functions
# ============================================================

ANCHOR_SYSTEMS = ['財務会計システム', 'HR人事給与システム', '受発注在庫管理システム', 'ProcMine SSO']
LABEL_TRANSLATIONS = {
    '財務会計システム': 'financial_accounting',
    'HR人事給与システム': 'hr_payroll',
    '受発注在庫管理システム': 'order_inventory',
    'ProcMine SSO': 'procmine_sso',
}
CLIPBOARD_MIN_GAP_S = 5
LONG_SEGMENT_THRESHOLD_MS = 60000
EXCEPTION_MULTIPLIER = 1.8
MIN_DURATION_FOR_VARIANT_S = 3  # skip variant tagging for very short segments


def load_session_events(session_dir):
    """Factor 3: load and merge all chunks in a session, sorted by real timestamp."""
    session_dir = Path(session_dir)
    all_events = []
    chunk_dirs = [d for d in session_dir.iterdir() if d.is_dir() and d.name.startswith("chunk_")]
    for chunk_dir in chunk_dirs:
        events_path = chunk_dir / "events.jsonl"
        if events_path.exists():
            with open(events_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        all_events.append(json.loads(line))
    all_events.sort(key=lambda e: e["timestamp_ms"])
    return all_events


def get_anchor(title):
    for a in ANCHOR_SYSTEMS:
        if a in title:
            return a
    return None


def filter_noise(app_switches_df):
    """Factor 1: drop consecutive app_switch events with same (process_id, window_title)."""
    kept, last = [], None
    for _, row in app_switches_df.iterrows():
        ident = (row['process_id'], row['window_title'])
        if ident != last:
            kept.append(row)
            last = ident
    return pd.DataFrame(kept)


def build_anchor_segments(clean_df, session_end_ms):
    """Factor 2 (primary): anchor-system based segmentation."""
    if len(clean_df) == 0:
        return []
    segments = []
    seg_start_ms = int(clean_df.iloc[0]['timestamp_iso'].timestamp() * 1000)
    current_anchor = None
    for i in range(len(clean_df)):
        title = clean_df.iloc[i]['window_title']
        anchor = get_anchor(title)
        if anchor is not None:
            if current_anchor is None:
                current_anchor = anchor
            elif anchor != current_anchor:
                seg_end_ms = int(clean_df.iloc[i]['timestamp_iso'].timestamp() * 1000)
                segments.append({'start_ms': seg_start_ms, 'end_ms': seg_end_ms, 'label': current_anchor})
                seg_start_ms = seg_end_ms
                current_anchor = anchor
    segments.append({'start_ms': seg_start_ms, 'end_ms': session_end_ms, 'label': current_anchor})
    return segments


def refine_with_clipboard(segments, clipboard_ts_list):
    """Factor 2 (secondary): split long segments using clipboard events."""
    refined = []
    for seg in segments:
        duration = seg['end_ms'] - seg['start_ms']
        label = seg.get('label', 'unknown')
        if duration <= LONG_SEGMENT_THRESHOLD_MS:
            refined.append(seg)
            continue
        inside = [t for t in clipboard_ts_list if seg['start_ms'] < t < seg['end_ms']]
        split_points = []
        last_kept = None
        for t in inside:
            if last_kept is None or (t - last_kept) > CLIPBOARD_MIN_GAP_S * 1000:
                split_points.append(t)
                last_kept = t
        if not split_points:
            refined.append(seg)
            continue
        prev = seg['start_ms']
        for sp in split_points:
            refined.append({'start_ms': prev, 'end_ms': sp, 'label': label})
            prev = sp
        refined.append({'start_ms': prev, 'end_ms': seg['end_ms'], 'label': label})
    return refined


def translate_labels(segments):
    """Factor 4: translate anchor labels to English."""
    for s in segments:
        s['label'] = LABEL_TRANSLATIONS.get(s['label'], s['label'])
    return segments


def tag_variants(segments, full_df):
    """Factor 7: flag unusually busy segments as '_exception' variants."""
    ts_ms_col = full_df['timestamp_iso'].apply(lambda t: int(t.timestamp() * 1000))

    def count_actions(start_ms, end_ms):
        mask = (ts_ms_col >= start_ms) & (ts_ms_col <= end_ms)
        subset = full_df[mask]
        counts = subset['event_type'].value_counts()
        return (counts.get('keystroke', 0) + counts.get('mouse_click', 0) +
                counts.get('shortcut', 0) + counts.get('mouse_scroll', 0))

    for seg in segments:
        duration_s = (seg['end_ms'] - seg['start_ms']) / 1000
        seg['action_density'] = count_actions(seg['start_ms'], seg['end_ms']) / duration_s if duration_s > 0 else 0
        seg['duration_s'] = duration_s

    density_by_label = defaultdict(list)
    for seg in segments:
        if seg['duration_s'] >= MIN_DURATION_FOR_VARIANT_S:
            density_by_label[seg['label']].append(seg['action_density'])

    median_density = {label: statistics.median(d) for label, d in density_by_label.items() if d}

    for seg in segments:
        if seg['duration_s'] < MIN_DURATION_FOR_VARIANT_S:
            continue
        med = median_density.get(seg['label'], 0)
        if med > 0 and seg['action_density'] > med * EXCEPTION_MULTIPLIER:
            seg['label'] = seg['label'] + '_exception'
    return segments


def process_session(session_dir, session_id):
    """
    Full pipeline: run all factors on one session, return final labeled segments
    in segments.jsonl-ready format.
    """
    raw_events = load_session_events(session_dir)
    if not raw_events:
        return []

    df = pd.DataFrame(raw_events)
    df['timestamp_iso'] = pd.to_datetime(df['timestamp_iso'])

    app_switches = df[df['event_type'] == 'app_switch'].copy()
    app_switches['process_id'] = app_switches['payload'].apply(lambda p: p['new_app']['process_id'])
    app_switches['window_title'] = app_switches['payload'].apply(lambda p: p['new_app']['window_title'])

    clean = filter_noise(app_switches).reset_index(drop=True)
    if len(clean) == 0:
        return []

    session_end_ms = int(df['timestamp_iso'].max().timestamp() * 1000)
    segments = build_anchor_segments(clean, session_end_ms)

    clipboard_events = df[df['event_type'] == 'clipboard_change'].copy()
    clipboard_ts_list = [int(t.timestamp() * 1000) for t in clipboard_events['timestamp_iso']]
    segments = refine_with_clipboard(segments, clipboard_ts_list)

    segments = translate_labels(segments)
    segments = tag_variants(segments, df)

    # Convert to segments.jsonl format
    output = []
    for s in segments:
        output.append({
            'session_id': session_id,
            'start': datetime.utcfromtimestamp(s['start_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),
            'end': datetime.utcfromtimestamp(s['end_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),
            'label': s['label'],
        })
    return output


from datetime import timezone

def evaluate_session(segments, gt_manifest_path):
    with open(gt_manifest_path, encoding='utf-8') as f:
        manifest = json.load(f)

    true_segments = []
    for proc in manifest['processes']:
        for ex in proc['executions']:
            if ex.get('start_ts') and ex.get('end_ts'):
                true_segments.append({'start_ts': ex['start_ts'], 'end_ts': ex['end_ts']})

    def to_ms(ts):
        return int(datetime.fromisoformat(ts).timestamp() * 1000)

    def iou_ms(ps, pe, ts, te):
        inter = max(0, min(pe, te) - max(ps, ts))
        union = max(pe, te) - min(ps, ts)
        return inter / union if union > 0 else 0.0

    pred_ms = []
    for s in segments:
        start_ms = int(datetime.strptime(s['start'], '%Y-%m-%dT%H:%M:%SZ').replace(tzinfo=timezone.utc).timestamp() * 1000)
        end_ms = int(datetime.strptime(s['end'], '%Y-%m-%dT%H:%M:%SZ').replace(tzinfo=timezone.utc).timestamp() * 1000)
        pred_ms.append({'start_ms': start_ms, 'end_ms': end_ms})

    scores = []
    for t in true_segments:
        t_start, t_end = to_ms(t['start_ts']), to_ms(t['end_ts'])
        best = max((iou_ms(p['start_ms'], p['end_ms'], t_start, t_end) for p in pred_ms), default=0.0)
        scores.append(best)

    return sum(scores) / len(scores) if scores else 0.0

# ============================================================
# RUN ACROSS ALL SESSIONS IN dataset_a
# ============================================================

dataset_a_path = Path("../data/dataset_a")
all_results = []
all_segments_output = []

session_dirs = sorted([d for d in dataset_a_path.iterdir() if d.is_dir()])
print(f"Found {len(session_dirs)} sessions")

for session_dir in session_dirs:
    session_id = session_dir.name
    try:
        segments = process_session(session_dir, session_id)
        all_segments_output.extend(segments)

        gt_manifest_path = session_dir / "gt_manifest.json"
        if gt_manifest_path.exists():
            avg_iou = evaluate_session(segments, gt_manifest_path)
            all_results.append({'session_id': session_id, 'avg_iou': avg_iou, 'n_segments': len(segments)})
            print(f"{session_id}: {len(segments)} segments, avg IoU = {avg_iou:.3f}")
    except Exception as e:
        print(f"ERROR on {session_id}: {e}")

overall_avg = sum(r['avg_iou'] for r in all_results) / len(all_results) if all_results else 0
print(f"\n=== OVERALL AVERAGE IoU across {len(all_results)} sessions: {overall_avg:.3f} ===")

# Show worst-performing sessions for your report's "what went wrong" analysis
print("\nWorst 5 sessions:")
for r in sorted(all_results, key=lambda r: r['avg_iou'])[:5]:
    print(f"  {r['session_id']}: {r['avg_iou']:.3f}")

Found 63 sessions


C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:183: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'start': datetime.utcfromtimestamp(s['start_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),
C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:184: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'end': datetime.utcfromtimestamp(s['end_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),


ses_20260630-121953-LAPTOP-R36BQBTE: 44 segments, avg IoU = 0.552


C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:183: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'start': datetime.utcfromtimestamp(s['start_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),
C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:184: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'end': datetime.utcfromtimestamp(s['end_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),
C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:183: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to re

ses_20260630-124826-CHAITANYA0BCF: 73 segments, avg IoU = 0.705
ses_20260630-125757-LAPTOP-R36BQBTE: 29 segments, avg IoU = 0.680


...

C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:183: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'start': datetime.utcfromtimestamp(s['start_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),
C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:184: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'end': datetime.utcfromtimestamp(s['end_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),
C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:183: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to re

ses_20260701-145907-MSI: 67 segments, avg IoU = 0.590
ses_20260701-152030-CHAITANYA0BCF: 25 segments, avg IoU = 0.202


C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:183: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'start': datetime.utcfromtimestamp(s['start_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),
C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:184: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'end': datetime.utcfromtimestamp(s['end_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),


ses_20260701-155031-CHAITANYA0BCF: 69 segments, avg IoU = 0.591
ses_20260701-172430-yuvraj: 51 segments, avg IoU = 0.734

=== OVERALL AVERAGE IoU across 63 sessions: 0.644 ===

Worst 5 sessions:
  ses_20260701-152030-CHAITANYA0BCF: 0.202
  ses_20260701-054901-LAPTOP-R36BQBTE: 0.464
  ses_20260701-043922-SIDDHIGUPTAB00B: 0.496
  ses_20260701-053005-MSI: 0.521
  ses_20260701-081732-CHAITANYA0BCF: 0.527


C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:183: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'start': datetime.utcfromtimestamp(s['start_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),
C:\Users\xlaks\AppData\Local\Temp\ipykernel_9212\2369158139.py:184: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'end': datetime.utcfromtimestamp(s['end_ms'] / 1000).strftime('%Y-%m-%dT%H:%M:%SZ'),
